# BazaarBot GRPO Training (Kaggle dual-T4)

End-to-end loop:
1. Auth into HF using Kaggle Secret `HF_TOKEN`.
2. Clone this project so we can import `bazaarbot_env`.
3. Load base model + tokenizer with 4-bit quant, attach LoRA adapters.
4. Define a reward function that runs `rollout_episode` per prompt.
5. Train with TRL's `GRPOTrainer`.
6. Push adapters to `PayMyBills/bestdealbot`.

**Before running:**
- Kaggle notebook settings → Accelerator: GPU T4 x2, Internet: On.
- Add-ons → Secrets → add `HF_TOKEN` (HF write token).
- Set `GITHUB_REPO` below to the pushed project URL.

**Quota:** one full run (~2–3 epochs over ~500 rollouts) fits inside Kaggle's 12-hour session.

## 1 · Install deps

In [ ]:
!pip install -q --upgrade "trl>=0.12" "peft>=0.13" "transformers>=4.46" "accelerate>=1.1" "bitsandbytes>=0.44" "datasets>=3.0" huggingface_hub

## 2 · HF auth (via Kaggle Secret)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = secret_value_0
login(token=secret_value_0)
print("HF login OK")

## 3 · Pull the bazaarbot_env package

Two options — pick one:
- **A** (recommended): push the repo to GitHub and clone it here.
- **B**: upload the `bazaarbot_env/` folder as a Kaggle Dataset and mount it.


In [ ]:
# Option A — git clone.  Replace with your repo URL once pushed.
GITHUB_REPO = "https://github.com/paymybills/BazaarBATNA.git"
!git clone {GITHUB_REPO} /kaggle/working/metathon

import sys
sys.path.insert(0, "/kaggle/working/metathon")

from bazaarbot_env import (
    BazaarGymEnv, rollout_episode, format_observation,
    parse_action, DEFAULT_SYSTEM_PROMPT, TASKS,
)

print("bazaarbot_env loaded. Tasks:", list(TASKS.keys()))

## 4 · Base model + LoRA

**Qwen3.5-4B** is the sweet spot: newest Qwen (Mar 2026), 5B effective params, fits
on one T4 with 4-bit + LoRA, produces valid action JSON out-of-the-box so GRPO
learns strategy rather than syntax, and quantizes to ~2.5GB GGUF for local Ollama
on your 2050.

Drop to `Qwen/Qwen3.5-2B` if rollouts OOM or you want `num_generations=8` for
lower-variance GRPO advantages. Qwen3.5 models are VLMs (Image-Text-to-Text) —
the vision encoder is unused during negotiation training, wasted capacity we'll
exploit later when we wire in listing images.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen3.5-4B"
REPO_ID    = "PayMyBills/bestdealbot"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5 · Prompt dataset

For GRPO we need prompts; the model generates N completions per prompt and the
reward function scores each.  Here a **prompt = (task, seed)** pair.  The reward
function samples a full episode per (prompt, completion) using that task/seed,
using the current model as the policy.

In [ ]:
import random
from datasets import Dataset

# amazon_realistic samples a real product + MRP/street-price per episode
# from data/amazon.csv (1417 unique listings).  Mixing with single_deal
# keeps the toy distribution as a sanity anchor.
TRAIN_TASKS = ["amazon_realistic", "amazon_realistic", "single_deal"]
N_PROMPTS = 256

rng = random.Random(0)
train_rows = []
for i in range(N_PROMPTS):
    task = rng.choice(TRAIN_TASKS)
    seed = rng.randint(0, 1_000_000)
    env = BazaarGymEnv(task_name=task, seed=seed)
    obs, _ = env.reset()
    user_turn = format_observation(obs)
    chat = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user",   "content": user_turn},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    train_rows.append({"prompt": chat, "task": task, "seed": seed})

train_ds = Dataset.from_list(train_rows)
print("Training prompts:", len(train_ds))
print("Sample task mix:", dict((t, sum(1 for r in train_rows if r['task'] == t)) for t in set(TRAIN_TASKS)))

## 6 · Reward function (full-episode rollout)

The reward for a single generated completion is the **terminal graded score of the full episode** played starting from that completion as the first action.  We run the rest of the episode with the same model.

This is expensive (N_rollouts × avg_steps forward passes per GRPO step) but correct —
the GRPO advantage now corresponds to "how good was the negotiation you produced"
rather than just "did you output valid JSON".

In [ ]:
@torch.no_grad()
def _generate(prompt_text: str, max_new_tokens: int = 64) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen_ids = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def reward_fn(completions, prompts=None, completion_ids=None, **kwargs):
    """GRPO reward: each completion becomes the first action; rollout continues."""
    rewards = []
    tasks = kwargs.get("task") or ["single_deal"] * len(completions)
    seeds = kwargs.get("seed") or [None] * len(completions)

    for completion, task, seed in zip(completions, tasks, seeds):
        env = BazaarGymEnv(task_name=task, seed=seed)
        obs, _ = env.reset()
        first_action = parse_action(completion, fallback_price=obs.get("own_private_budget", 100) * 0.3)

        parse_penalty = -0.2 if first_action.get("_parse_error") else 0.0
        _obs, r, done, info = env.step(first_action)

        # Continue with the current policy to terminal.
        history = []
        max_turns = 12
        while not done and max_turns > 0:
            user = format_observation(_obs, history=history)
            chat = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                    {"role": "user",   "content": user},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            raw = _generate(chat)
            act = parse_action(raw, fallback_price=_obs.get("own_private_budget", 100) * 0.3)
            _obs, r, done, info = env.step(act)
            max_turns -= 1

        rewards.append(float(env.score()) + parse_penalty)
    return rewards

## 7 · GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# TRL's GRPOConfig API changes across versions; max_prompt_length was removed
# in newer releases.  If you get "unexpected keyword argument", comment out
# any offending fields or run `print(GRPOConfig.__dataclass_fields__.keys())`
# to see what your installed version accepts.
grpo_cfg = GRPOConfig(
    output_dir="/kaggle/working/bestdealbot-ckpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=64,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=50,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_cfg,
    train_dataset=train_ds,
)

trainer.train()

## 8 · Push adapters to HF

Only the LoRA adapters get pushed (~50MB), not the merged model.  Merge locally
after download to avoid Kaggle network bottleneck.

In [ ]:
model.save_pretrained("/kaggle/working/bestdealbot-adapter")
tokenizer.save_pretrained("/kaggle/working/bestdealbot-adapter")

from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/kaggle/working/bestdealbot-adapter",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="GRPO adapter from Kaggle run",
)
print(f"Pushed to https://huggingface.co/{REPO_ID}")

## Local post-training

```bash
# pull adapter
hf download PayMyBills/bestdealbot --local-dir ~/models/bdb

# merge base + adapter (run on your laptop)
python -m peft merge_and_unload \
    --base-model Qwen/Qwen3.5-4B \
    --adapter  ~/models/bdb \
    --out      ~/models/bdb-merged

# convert to GGUF
python llama.cpp/convert_hf_to_gguf.py ~/models/bdb-merged \
    --outfile bdb.gguf --outtype q4_k_m

# register with Ollama
cat > Modelfile <<'EOF'
FROM ./bdb.gguf
SYSTEM """You are a skilled buyer negotiating at an Indian bazaar."""
EOF
ollama create bestdealbot -f Modelfile
ollama run bestdealbot
```